In [4]:
import pandas as pd
import numpy as np

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import f1_score
import ruptures as rpt


In [ ]:
from pathlib import Path
DATA_DIR = Path('data')


In [ ]:
demand_raw = pd.read_csv(DATA_DIR / 'demand_history.csv')

month_cols = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

demand = demand_raw.melt(
    id_vars=["product", "aps", "year"],
    value_vars=month_cols,
    var_name="month",
    value_name="demand"
)

demand["date"] = pd.to_datetime(
    demand["year"].astype(str) + "-" + demand["month"] + "-01"
)

demand = demand.rename(columns={"product": "product_id"})
demand = demand.sort_values(["product_id", "aps", "date"]).reset_index(drop=True)


In [6]:
labels = pd.read_csv(DATA_DIR / 'anomaly_labels.csv')

labels["date"] = pd.to_datetime(
    labels["year"].astype(str) + "-" + labels["month"] + "-01"
)

labels = labels.rename(columns={"product": "product_id"})
labels["is_anomaly"] = labels["is_anomaly"].astype(bool).astype(int)

labels = labels.sort_values(["product_id", "aps", "date"]).reset_index(drop=True)


In [7]:
promo = pd.read_csv(DATA_DIR / 'promotion_history.csv')

promo["date"] = pd.to_datetime(
    promo["year"].astype(str) + "-" + promo["month"] + "-01"
)

promo = promo.rename(columns={"product": "product_id"})


In [8]:
capacity = pd.read_csv(DATA_DIR / 'capacity_constraints.csv')

capacity["date"] = pd.to_datetime(
    capacity["year"].astype(str) + "-" +
    capacity["month"].astype(str).str.zfill(2) + "-01"
)

capacity = capacity.rename(columns={"product": "product_id"})


In [9]:
def forecast_next(series):
    model = SARIMAX(
        series,
        order=(1,1,1),
        seasonal_order=(1,1,1,12),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    res = model.fit(disp=False)

    fc = res.get_forecast(steps=1)
    pred = fc.predicted_mean.iloc[0]
    ci = fc.conf_int(alpha=0.10).iloc[0]

    return pred, ci[0], ci[1]


In [10]:
def anomaly_score(actual, lower, upper):
    if actual > upper:
        return actual - upper, "spike"
    elif actual < lower:
        return lower - actual, "drop"
    else:
        return 0.0, "normal"


In [11]:
def detect_change_points(series):
    algo = rpt.Pelt(model="rbf").fit(series.values)
    return algo.predict(pen=5)


In [12]:
def rolling_baseline(series, window=12):
    median = series.rolling(window, min_periods=6).median()
    mad = (series - median).abs().rolling(window, min_periods=6).median()
    return median, mad


In [13]:
def detect_point_anomaly(actual, median, mad, k=3):
    if pd.isna(median) or pd.isna(mad) or mad == 0:
        return 0, "normal"

    z = (actual - median) / (1.4826 * mad)

    if z > k:
        return z, "spike"
    elif z < -k:
        return abs(z), "drop"
    else:
        return 0, "normal"


In [14]:
records = []

for (pid, aps), g in demand.groupby(["product_id", "aps"]):
    g = g.sort_values("date").reset_index(drop=True)

    if len(g) < 12:
        continue

    median, mad = rolling_baseline(g["demand"])

    change_points = detect_change_points(g["demand"])

    for i in range(len(g)):
        score, direction = detect_point_anomaly(
            g.loc[i, "demand"],
            median.iloc[i],
            mad.iloc[i]
        )

        anomaly_type = None

        if score > 0:
            anomaly_type = direction

        if i in change_points:
            anomaly_type = "trend_break"

        if anomaly_type:
            records.append({
                "product_id": pid,
                "aps": aps,
                "date": g.loc[i, "date"],
                "actual": g.loc[i, "demand"],
                "baseline": median.iloc[i],
                "score": score,
                "direction": direction,
                "anomaly_type_pred": anomaly_type
            })

anomalies = pd.DataFrame(records)


In [15]:
def detect_anomalies_with_k(demand, k):
    records = []

    for (pid, aps), g in demand.groupby(["product_id", "aps"]):
        g = g.sort_values("date").reset_index(drop=True)

        if len(g) < 12:
            continue

        median = g["demand"].rolling(12, min_periods=6).median()
        mad = (g["demand"] - median).abs().rolling(12, min_periods=6).median()

        for i in range(len(g)):
            if pd.isna(median.iloc[i]) or pd.isna(mad.iloc[i]) or mad.iloc[i] == 0:
                continue

            z = (g.loc[i, "demand"] - median.iloc[i]) / (1.4826 * mad.iloc[i])

            if abs(z) > k:
                records.append({
                    "product_id": pid,
                    "aps": aps,
                    "date": g.loc[i, "date"]
                })

    # 🔑 CRITICAL FIX: return empty DF with correct columns
    return pd.DataFrame(records, columns=["product_id", "aps", "date"])


In [16]:
from sklearn.metrics import f1_score

def compute_f1(pred_anoms, demand, labels):
    eval_df = demand.merge(
        labels[["product_id", "aps", "date", "is_anomaly"]],
        on=["product_id", "aps", "date"],
        how="left"
    )

    eval_df["is_anomaly"] = eval_df["is_anomaly"].fillna(0).astype(int)
    eval_df["predicted"] = 0

    # Only mark predictions if any exist
    if not pred_anoms.empty:
        eval_df.loc[
            eval_df.set_index(["product_id", "aps", "date"]).index.isin(
                pred_anoms.set_index(["product_id", "aps", "date"]).index
            ),
            "predicted"
        ] = 1

    # Avoid undefined F1
    if eval_df["predicted"].sum() == 0:
        return 0.0

    return f1_score(eval_df["is_anomaly"], eval_df["predicted"])


In [17]:
k_values = np.arange(2.0, 4.5, 0.25)

results = []

for k in k_values:
    pred_anoms = detect_anomalies_with_k(demand, k)
    f1 = compute_f1(pred_anoms, demand, labels)
    results.append((k, f1))
    print(f"k={k:.2f} → F1={f1:.4f}")


k=2.00 → F1=0.0784
k=2.25 → F1=0.0455
k=2.50 → F1=0.0000
k=2.75 → F1=0.0000
k=3.00 → F1=0.0000
k=3.25 → F1=0.0000
k=3.50 → F1=0.0000
k=3.75 → F1=0.0000
k=4.00 → F1=0.0000
k=4.25 → F1=0.0000


In [18]:
def detect_trend_breaks(series, window=6, slope_thresh=2.0):
    slopes = series.diff().rolling(window).mean()
    slope_change = slopes.diff().abs()

    return slope_change > slope_thresh * slope_change.median()


In [19]:
def detect_pattern_shift(series, short=6, long=12, ratio=1.3):
    short_mean = series.rolling(short).mean()
    long_mean = series.rolling(long).mean()

    shift = (short_mean - long_mean).abs() / long_mean.abs()
    return shift > ratio


In [20]:
records = []

for (pid, aps), g in demand.groupby(["product_id", "aps"]):
    g = g.sort_values("date").reset_index(drop=True)

    if len(g) < 12:
        continue

    # Point anomaly baseline
    median = g["demand"].rolling(12, min_periods=6).median()
    mad = (g["demand"] - median).abs().rolling(12, min_periods=6).median()

    # Structural detectors
    trend_breaks = detect_trend_breaks(g["demand"])
    pattern_shifts = detect_pattern_shift(g["demand"])

    for i in range(len(g)):
        date = g.loc[i, "date"]
        actual = g.loc[i, "demand"]

        anomaly_type = None
        direction = None
        score = 0

        # -------- POINT ANOMALY --------
        if not pd.isna(median.iloc[i]) and not pd.isna(mad.iloc[i]) and mad.iloc[i] > 0:
            z = (actual - median.iloc[i]) / (1.4826 * mad.iloc[i])

            if z > 2.0:
                anomaly_type = "spike"
                direction = "up"
                score = z
            elif z < -2.0:
                anomaly_type = "drop"
                direction = "down"
                score = abs(z)

        # -------- STRUCTURAL OVERRIDE --------
        if trend_breaks.iloc[i]:
            anomaly_type = "trend_break"
            direction = "structural"

        if pattern_shifts.iloc[i]:
            anomaly_type = "pattern_shift"
            direction = "structural"

        if anomaly_type:
            records.append({
                "product_id": pid,
                "aps": aps,
                "date": date,
                "actual": actual,
                "anomaly_type_pred": anomaly_type,
                "direction": direction,
                "score": score
            })

anomalies = pd.DataFrame(
    records,
    columns=["product_id", "aps", "date", "actual",
             "anomaly_type_pred", "direction", "score"]
)


In [21]:
f1 = compute_f1(anomalies[["product_id", "aps", "date"]], demand, labels)
print("🔥 FINAL F1:", round(f1, 4))


🔥 FINAL F1: 0.1205


In [22]:
def enforce_persistence(flags, min_len=2):
    persistent = flags.copy() * False
    count = 0

    for i, val in enumerate(flags):
        if val:
            count += 1
        else:
            count = 0

        if count >= min_len:
            persistent.iloc[i] = True

    return persistent


In [23]:
raw_trend = detect_trend_breaks(g["demand"])
raw_pattern = detect_pattern_shift(g["demand"])

trend_breaks = enforce_persistence(raw_trend, min_len=2)
pattern_shifts = enforce_persistence(raw_pattern, min_len=2)


In [24]:
cooldown = 0

for i in range(len(g)):
    if cooldown > 0:
        cooldown -= 1
        continue

    anomaly_type = None
    direction = None
    score = 0

    # ----- point anomaly -----
    if not pd.isna(median.iloc[i]) and not pd.isna(mad.iloc[i]) and mad.iloc[i] > 0:
        z = (g.loc[i, "demand"] - median.iloc[i]) / (1.4826 * mad.iloc[i])
        if z > 2.0:
            anomaly_type = "spike"
            score = z
        elif z < -2.0:
            anomaly_type = "drop"
            score = abs(z)

    # ----- structural override -----
    if trend_breaks.iloc[i]:
        anomaly_type = "trend_break"
        cooldown = 3

    if pattern_shifts.iloc[i]:
        anomaly_type = "pattern_shift"
        cooldown = 3

    if anomaly_type:
        records.append({
            "product_id": pid,
            "aps": aps,
            "date": g.loc[i, "date"],
            "actual": g.loc[i, "demand"],
            "anomaly_type_pred": anomaly_type,
            "score": score
        })


In [25]:
f1 = compute_f1(anomalies[["product_id", "aps", "date"]], demand, labels)
print("🔥 FINAL F1 (precision-tuned):", round(f1, 4))


🔥 FINAL F1 (precision-tuned): 0.1205


In [26]:
label_events = labels[labels.is_anomaly == 1][
    ["product_id", "aps", "date"]
].copy()


In [27]:
def event_based_f1(pred_df, label_df, window=1):
    matched_labels = set()
    matched_preds = set()

    for li, l in label_df.iterrows():
        lkey = (l.product_id, l.aps, l.date)

        candidates = pred_df[
            (pred_df.product_id == l.product_id) &
            (pred_df.aps == l.aps) &
            (pred_df.date.between(
                l.date - pd.DateOffset(months=window),
                l.date + pd.DateOffset(months=window)
            ))
        ]

        if not candidates.empty:
            matched_labels.add(lkey)
            matched_preds.add(candidates.index[0])

    tp = len(matched_labels)
    fp = len(pred_df) - len(matched_preds)
    fn = len(label_df) - len(matched_labels)

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

    return f1, precision, recall


In [28]:
import pandas as pd

demand = pd.read_csv(DATA_DIR / 'demand_history.csv')

print(demand.columns)
print(demand.head())


Index(['product', 'aps', 'year', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
       'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
      dtype='object')
  product     aps  year    Jan    Feb    Mar    Apr    May    Jun    Jul  \
0      AH    ACNF  2022  10733  10328  11188  13739  14420  14603  16511   
1      AH    ACNF  2023  10986  10045  12266  14883  15757  16825  18040   
2      AH    ACNF  2024  11780  11171  12055  15155  16929  17593  19231   
3      AH    ACNF  2025  12858  10793  13054  16149  18015  17905  19817   
4      AH  AH_FIT  2022   7499   6928   7445   8990   9704  10243  11236   

     Aug    Sep    Oct    Nov    Dec  
0  14448  15313  14503  11708   8903  
1  15043  15044  13479  13493  10554  
2  17170  16159  13870  11529  12644  
3  17575  17790  15962  13540  11776  
4   9334  10446   9331   7715   6057  


In [29]:
MONTH_MAP = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

month_cols = list(MONTH_MAP.keys())

demand_long = demand.melt(
    id_vars=["product", "aps", "year"],
    value_vars=month_cols,
    var_name="month",
    value_name="demand"
)

demand_long["month_num"] = demand_long["month"].map(MONTH_MAP)

demand_long["date"] = pd.to_datetime(
    demand_long["year"].astype(str) + "-" +
    demand_long["month_num"].astype(str) + "-01"
)

demand_long = demand_long.rename(columns={"product": "product_id"})


In [30]:
print(demand_long.head())
print(demand_long.dtypes)


  product_id     aps  year month  demand  month_num       date
0         AH    ACNF  2022   Jan   10733          1 2022-01-01
1         AH    ACNF  2023   Jan   10986          1 2023-01-01
2         AH    ACNF  2024   Jan   11780          1 2024-01-01
3         AH    ACNF  2025   Jan   12858          1 2025-01-01
4         AH  AH_FIT  2022   Jan    7499          1 2022-01-01
product_id            object
aps                   object
year                   int64
month                 object
demand                 int64
month_num              int64
date          datetime64[ns]
dtype: object


In [31]:
import numpy as np

def detect_anomalies_with_k(df, k=2.0, min_history=6):
    """
    Rolling statistical anomaly detector
    """
    anomalies = []

    df = df.sort_values(["product_id", "aps", "date"])

    for (pid, aps), g in df.groupby(["product_id", "aps"]):
        g = g.reset_index(drop=True)

        for i in range(min_history, len(g)):
            history = g.loc[:i-1, "demand"]

            mean = history.mean()
            std = history.std()

            if std == 0 or np.isnan(std):
                continue

            upper = mean + k * std
            lower = mean - k * std
            actual = g.loc[i, "demand"]

            if actual > upper or actual < lower:
                anomalies.append({
                    "product_id": pid,
                    "aps": aps,
                    "date": g.loc[i, "date"],
                    "actual": actual,
                    "lower": lower,
                    "upper": upper,
                    "direction": "up" if actual > upper else "down"
                })

    return pd.DataFrame(anomalies)

                    


In [32]:
anomalies = detect_anomalies_with_k(demand_long, k=2.0)

print("Detected anomalies:", len(anomalies))
print(anomalies.head())


Detected anomalies: 52
  product_id   aps       date  actual        lower         upper direction
0         AH  ACNF 2022-07-01   16511  8582.106115  16421.560552        up
1         AH  ACNF 2022-12-01    8903  9282.510519  17534.580390      down
2         AH  ACNF 2023-07-01   18040  8331.297680  18019.702320        up
3         AH  ACNF 2024-07-01   19231  8639.402442  18526.930892        up
4         AH  ACNF 2025-07-01   19817  8738.320996  19207.964718        up


In [33]:
labels = pd.read_csv(DATA_DIR / 'anomaly_labels.csv')

labels_raw = pd.read_csv(DATA_DIR / 'anomaly_labels.csv')

print(labels_raw.columns)
print(labels_raw.head())


Index(['product', 'aps', 'year', 'month', 'is_anomaly', 'anomaly_type',
       'magnitude', 'root_cause'],
      dtype='object')
  product     aps  year month  is_anomaly      anomaly_type  magnitude  \
0      AH    ACNF  2022   Dec        True     drop_shortage      -21.3   
1      CL   CL_MB  2022   Feb        True       trend_break      -17.4   
2      HP  HP_3PH  2022   Jan        True       spike_promo       16.9   
3      CL   CL_MB  2022   Oct        True       spike_promo       26.2   
4      HP  HP_3PH  2022   Apr        True  drop_competition      -12.9   

                      root_cause  
0             Component shortage  
1  Construction cycle inflection  
2    Seasonal promotion campaign  
3    Seasonal promotion campaign  
4        Competitor pricing move  


In [34]:
MONTH_MAP = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

labels_raw["month_num"] = labels_raw["month"].map(MONTH_MAP)

labels_raw["date"] = pd.to_datetime(
    labels_raw["year"].astype(str) + "-" +
    labels_raw["month_num"].astype(str) + "-01"
)

label_events = labels_raw.rename(columns={
    "product": "product_id"
})

label_events = label_events[
    label_events["is_anomaly"] == True
][["product_id", "aps", "date"]].drop_duplicates()


In [35]:
print(label_events.columns)
print(label_events.head())
print("Total labeled events:", len(label_events))


Index(['product_id', 'aps', 'date'], dtype='object')
  product_id     aps       date
0         AH    ACNF 2022-12-01
1         CL   CL_MB 2022-02-01
2         HP  HP_3PH 2022-01-01
3         CL   CL_MB 2022-10-01
4         HP  HP_3PH 2022-04-01
Total labeled events: 40


In [36]:
import pandas as pd
import numpy as np

def event_based_f1(pred_events, true_events, window=1):
    """
    Event-based F1 score with temporal tolerance.

    pred_events: DataFrame with columns [product_id, aps, date]
    true_events: DataFrame with columns [product_id, aps, date]
    window: number of months tolerance (+/-)

    Returns: f1, precision, recall
    """

    if len(pred_events) == 0 or len(true_events) == 0:
        return 0.0, 0.0, 0.0

    # Ensure datetime
    pred = pred_events.copy()
    true = true_events.copy()

    pred["date"] = pd.to_datetime(pred["date"])
    true["date"] = pd.to_datetime(true["date"])

    matched_true = set()
    tp = 0

    for _, p in pred.iterrows():
        candidates = true[
            (true["product_id"] == p["product_id"]) &
            (true["aps"] == p["aps"]) &
            (abs((true["date"] - p["date"]).dt.days) <= window * 31)
        ]

        if len(candidates) > 0:
            idx = candidates.index[0]
            if idx not in matched_true:
                tp += 1
                matched_true.add(idx)

    fp = len(pred) - tp
    fn = len(true) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    return f1, precision, recall


In [ ]:
f1, precision, recall = event_based_f1(
    anomalies,
    label_events,
    window=1
)

print("EVENT-BASED F1:", round(f1, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
